# 🎬 AI Video Upscaler 4K - Anime Native 2x WebUI (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhSi1008/ai-video-upscaler/blob/main/notebooks/Google_Colab_RealCUGAN_4K.ipynb)

Hệ thống nâng cấp Anime từ 1080p lên **4K Ultra-HD (3840x2160 Native 2x)** sử dụng giao diện WebUI trực quan.
- **⚡ 2 Mô hình chuyên biệt:** Nguồn B (WEB-DL Gốc) & Nguồn A (BDRip 10-bit Hi10P/Main 10).
- **Giao diện WebUI:** Dán Link Google Drive hoặc chọn tệp video trực tiếp trên trình duyệt.
- **Tự động lưu Drive & Checkpoint:** Video 4K hoàn chỉnh được lưu vĩnh viễn vào Google Drive (`/content/drive/MyDrive/Upscaled`), tự động tiếp tục khi mất kết nối.
- **Mã hóa chuyên nghiệp:** HEVC 10-bit (`yuv420p10le`), giữ nguyên 100% âm thanh và phụ đề mềm (.ass).
- **Tốc độ siêu tốc:** ~12-15 FPS trên GPU NVIDIA Tesla T4 (~25-40 phút/tập 24 phút).

---

In [ ]:
# @title 🚀 BƯỚC 1: Kết Nối Google Drive & Kiểm Tra GPU T4
# @markdown Bấm nút Run (Ctrl+Enter) và chọn "Kết nối với Google Drive" khi bảng hỏi hiện lên.
from google.colab import drive
import torch
import os

drive.mount('/content/drive')

# Tự động tạo sẵn thư mục Resources và Upscaled trên Drive
os.makedirs('/content/drive/MyDrive/Resources', exist_ok=True)
os.makedirs('/content/drive/MyDrive/Upscaled', exist_ok=True)

print("\n" + "="*60)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"✅ ĐÃ SẴN SÀNG GPU TỐC ĐỘ CAO: {gpu_name}")
    print(f"📁 Thư mục nguồn sẵn sàng: /content/drive/MyDrive/Resources/")
    print(f"📁 Thư mục xuất 4K sẵn sàng: /content/drive/MyDrive/Upscaled/")
else:
    print("⚠️ CHƯA BẬT GPU! Vào menu: Runtime -> Change runtime type -> T4 GPU")
print("="*60)

In [ ]:
# @title 📦 BƯỚC 2: Cài Đặt Môi Trường & Kích Hoạt Tăng Tốc NVIDIA NVENC
# @markdown Cài đặt FFmpeg NVENC (NVIDIA Hardware Fast 12-15+ FPS), Gradio và đồng bộ mã nguồn (~30s).
import os

# Tự động cài đặt bản build FFmpeg hỗ trợ NVIDIA NVENC phần cứng siêu tốc
!if ! ffmpeg -encoders 2>/dev/null | grep -q "hevc_nvenc"; then \
    echo "⚡ Đang tải và cài đặt FFmpeg hỗ trợ NVIDIA NVENC..."; \
    (wget -q --timeout=30 https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz -O /tmp/ffmpeg.tar.xz && \
     tar -xf /tmp/ffmpeg.tar.xz -C /tmp && \
     cp -f /tmp/ffmpeg-*-linux64-gpl/bin/ffmpeg /usr/local/bin/ffmpeg && \
     cp -f /tmp/ffmpeg-*-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe && \
     chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe && \
     rm -rf /tmp/ffmpeg*) || (apt-get update -qq && apt-get install -y ffmpeg -qq); \
fi

!pip install -q --no-cache-dir gradio opencv-python gdown

repo_dir = "/content/ai-video-upscaler"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git reset --hard -q
    !git pull
else:
    !git clone https://github.com/ThanhSi1008/ai-video-upscaler.git {repo_dir}
    %cd {repo_dir}

print("\n" + "="*60)
!ffmpeg -encoders 2>&1 | grep -E "hevc_nvenc|libx265" || true
print("="*60)
print("✅ ĐÃ ĐỒNG BỘ MÃ NGUỒN VÀ MÔI TRƯỜNG TĂNG TỐC THÀNH CÔNG!")

In [ ]:
# @title 🌐 BƯỚC 3: Khởi Chạy Giao Diện WebUI 4K
# @markdown Bấm Run ô này, sau đó nhấp vào đường link Public Gradio (ví dụ: `https://xxxx.gradio.live`) để mở giao diện WebUI trên trình duyệt:
import os
repo_dir = "/content/ai-video-upscaler"
if os.path.exists(repo_dir):
    %cd {repo_dir}
    !git pull
else:
    !git clone https://github.com/ThanhSi1008/ai-video-upscaler.git {repo_dir}
    %cd {repo_dir}

!fuser -k 7860/tcp 2>/dev/null || true
!python3 -u app.py --share